# Agent: People (Phase 1, no MQTT)

This notebook implements a minimal local simulation for 50 people around Brøndby Stadium.

Phase 1 scope:
- One agent only
- Basic movement and state transitions
- No MQTT publish/subscribe
- Configuration loaded with `simulated_city.config.load_config()`

In [ ]:
# Cell purpose: import required modules and load project configuration
from __future__ import annotations

from dataclasses import dataclass
import math
import random
from typing import Literal

from simulated_city.config import load_config

cfg = load_config()
sim_cfg = cfg.simulation

print(f"Loaded config. MQTT base topic: {cfg.mqtt.base_topic}")
print(f"Simulation section present: {sim_cfg is not None}")

In [ ]:
# Cell purpose: define local data model for people state used in Phase 1
State = Literal["approaching_entry", "inside", "exiting", "exited"]

@dataclass
class PersonState:
    person_id: str
    name: str
    lat: float
    lon: float
    state: State
    color: Literal["white", "green", "red"]
    speed_mps: float
    target_entry_id: str
    target_exit_id: str

print("Defined PersonState dataclass for local simulation state.")

In [ ]:
# Cell purpose: set simulation constants and build entry/exit points from config with safe fallbacks
TOTAL_PEOPLE = 50
TICKS = 180
TICK_SECONDS = 1.0
ENTRY_PROXIMITY_M = 8.0
EXIT_REACHED_M = 8.0
TRUE_ALLOW_PROBABILITY = 0.80
SEED = sim_cfg.seed if sim_cfg and sim_cfg.seed is not None else 42

# Fallback center near Brøndby Stadium if no simulation locations are configured
center_lat = 55.6479
center_lon = 12.0417

configured_locations = list(sim_cfg.locations) if sim_cfg and sim_cfg.locations else []

if configured_locations:
    entries = [
        {"entry_id": loc.location_id, "lat": loc.lat, "lon": loc.lon}
        for loc in configured_locations
    ]
else:
    entries = [
        {"entry_id": "E1", "lat": center_lat + 0.0014, "lon": center_lon - 0.0009},
        {"entry_id": "E2", "lat": center_lat + 0.0012, "lon": center_lon + 0.0011},
        {"entry_id": "E3", "lat": center_lat - 0.0012, "lon": center_lon + 0.0010},
        {"entry_id": "E4", "lat": center_lat - 0.0013, "lon": center_lon - 0.0011},
    ]

exit_waypoints = [
    {"waypoint_id": "X1", "lat": center_lat + 0.0030, "lon": center_lon - 0.0030},
    {"waypoint_id": "X2", "lat": center_lat + 0.0030, "lon": center_lon + 0.0030},
    {"waypoint_id": "X3", "lat": center_lat - 0.0030, "lon": center_lon + 0.0030},
    {"waypoint_id": "X4", "lat": center_lat - 0.0030, "lon": center_lon - 0.0030},
]

name_pool = [
    "Alex", "Maja", "Jonas", "Emma", "Noah", "Ida", "Lucas", "Freja", "Oscar", "Sofia",
    "Viktor", "Alma", "William", "Clara", "Malthe", "Nora", "Anton", "Liva", "Elias", "Agnes"
]

print(f"Entries configured: {len(entries)}")
print(f"Exit waypoints configured: {len(exit_waypoints)}")
print(f"Using random seed: {SEED}")

In [ ]:
# Cell purpose: define movement helpers (distance, step movement, nearest target lookups)
def meters_between(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    earth_radius_m = 6_371_000.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return earth_radius_m * c

def move_towards(lat: float, lon: float, target_lat: float, target_lon: float, step_m: float) -> tuple[float, float]:
    distance_m = meters_between(lat, lon, target_lat, target_lon)
    if distance_m <= 1e-9:
        return lat, lon
    ratio = min(1.0, step_m / distance_m)
    new_lat = lat + (target_lat - lat) * ratio
    new_lon = lon + (target_lon - lon) * ratio
    return new_lat, new_lon

def nearest_point(lat: float, lon: float, points: list[dict], id_key: str) -> dict:
    return min(points, key=lambda point: (meters_between(lat, lon, point["lat"], point["lon"]), point[id_key]))

print("Movement helper functions are ready.")

In [ ]:
# Cell purpose: initialize 50 people with white status and dynamic nearest-entry routing
rng = random.Random(SEED)
people: list[PersonState] = []

for index in range(TOTAL_PEOPLE):
    start_lat = center_lat + rng.uniform(-0.0022, 0.0022)
    start_lon = center_lon + rng.uniform(-0.0022, 0.0022)
    nearest_entry = nearest_point(start_lat, start_lon, entries, "entry_id")
    nearest_exit = nearest_point(start_lat, start_lon, exit_waypoints, "waypoint_id")

    person = PersonState(
        person_id=f"p{index + 1:03d}",
        name=name_pool[index % len(name_pool)],
        lat=start_lat,
        lon=start_lon,
        state="approaching_entry",
        color="white",
        speed_mps=rng.uniform(0.8, 1.8),
        target_entry_id=nearest_entry["entry_id"],
        target_exit_id=nearest_exit["waypoint_id"],
    )
    people.append(person)

print(f"Initialized {len(people)} people.")
print(f"Initial white count: {sum(1 for person in people if person.color == 'white')}")

In [ ]:
# Cell purpose: run simulation loop and print deterministic summary for Phase 1 verification
simulation_log: list[dict] = []
inside_count = 0

for tick in range(TICKS):
    for person in people:
        if person.state == "inside" or person.state == "exited":
            continue

        if person.state == "approaching_entry":
            entry = next(entry_item for entry_item in entries if entry_item["entry_id"] == person.target_entry_id)
            person.lat, person.lon = move_towards(
                person.lat, person.lon, entry["lat"], entry["lon"], person.speed_mps * TICK_SECONDS
            )
            distance_to_entry = meters_between(person.lat, person.lon, entry["lat"], entry["lon"])

            if distance_to_entry <= ENTRY_PROXIMITY_M:
                is_allowed = rng.random() < TRUE_ALLOW_PROBABILITY
                if is_allowed:
                    person.state = "inside"
                    person.color = "green"
                    inside_count += 1
                else:
                    person.state = "exiting"
                    person.color = "red"

        elif person.state == "exiting":
            exit_point = next(point for point in exit_waypoints if point["waypoint_id"] == person.target_exit_id)
            person.lat, person.lon = move_towards(
                person.lat, person.lon, exit_point["lat"], exit_point["lon"], person.speed_mps * TICK_SECONDS
            )
            distance_to_exit = meters_between(person.lat, person.lon, exit_point["lat"], exit_point["lon"])
            if distance_to_exit <= EXIT_REACHED_M:
                person.state = "exited"

    white_count = sum(1 for person in people if person.color == "white")
    green_count = sum(1 for person in people if person.color == "green")
    red_count = sum(1 for person in people if person.color == "red")

    simulation_log.append(
        {
            "tick": tick,
            "white": white_count,
            "green": green_count,
            "red": red_count,
            "inside_count": inside_count,
        }
    )

final_white = simulation_log[-1]["white"]
final_green = simulation_log[-1]["green"]
final_red = simulation_log[-1]["red"]
final_exited = sum(1 for person in people if person.state == "exited")

print("Phase 1 simulation complete.")
print(f"Final counts -> white={final_white}, green={final_green}, red={final_red}, exited={final_exited}, inside_count={inside_count}")
print("Sample records (first 3 people):")
for person in people[:3]:
    print({
        "person_id": person.person_id,
        "name": person.name,
        "state": person.state,
        "color": person.color,
        "target_entry_id": person.target_entry_id,
    })